# Sistemas de Recomendación para la Oficina del Censo de EE.UU.

In [9]:
# Cargar dataset
import pandas as pd

# URL del dataset proporcionado
url = "https://breathecode.herokuapp.com/asset/internal-link?id=2326&path=adult-census-income.csv"

# Cargar el dataset
df = pd.read_csv(url)

# Limpiar espacios en los nombres de columnas (por si acaso)
df.columns = df.columns.str.strip()

# Seleccionar solo las columnas deseadas
cols = [
    'age',
    'education',
    'marital.status',
    'occupation',
    'hours.per.week',
    'sex',
    'native.country',
    'income'
]

df_selected = df[cols]

# Mostrar primeras filas
df_selected.head()


,age,education,marital.status,occupation,hours.per.week,sex,native.country,income
0,90,HS-grad,Widowed,?,40,Female,United-States,<=50K
1,82,HS-grad,Widowed,Exec-managerial,18,Female,United-States,<=50K
2,66,Some-college,Widowed,?,40,Female,United-States,<=50K
3,54,7th-8th,Divorced,Machine-op-inspct,40,Female,United-States,<=50K
4,41,Some-college,Separated,Prof-specialty,40,Female,United-States,<=50K


In [10]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

# Cargar dataset
url = "https://breathecode.herokuapp.com/asset/internal-link?id=2326&path=adult-census-income.csv"
df = pd.read_csv(url)

# Limpiar nombres de columnas
df.columns = df.columns.str.strip()

# Seleccionar columnas relevantes
cols = [
    'age',
    'education',
    'marital.status',
    'occupation',
    'hours.per.week',
    'sex',
    'native.country',
    'income'
]

df = df[cols]

# ---------------------------------------------------------
# 1. LIMPIEZA DE DATOS MAL CODIFICADOS
# ---------------------------------------------------------

# Reemplazar '?' por NaN
df = df.replace('?', pd.NA)

# Eliminar filas con valores faltantes
df = df.dropna()

# ---------------------------------------------------------
# 2. SEPARAR VARIABLES Y TARGET
# ---------------------------------------------------------

X = df.drop('income', axis=1)
y = df['income']

# ---------------------------------------------------------
# 3. IDENTIFICAR VARIABLES NUMÉRICAS Y CATEGÓRICAS
# ---------------------------------------------------------

numeric_features = ['age', 'hours.per.week']
categorical_features = [
    'education',
    'marital.status',
    'occupation',
    'sex',
    'native.country'
]

# ---------------------------------------------------------
# 4. TRANSFORMACIONES: OneHot + Normalización
# ---------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

# ---------------------------------------------------------
# 5. DIVISIÓN TRAIN / TEST
# ---------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---------------------------------------------------------
# 6. PIPELINE FINAL (solo preprocesamiento)
# ---------------------------------------------------------

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor)
])

# Transformar los datos
X_train_processed = pipeline.fit_transform(X_train)
X_test_processed = pipeline.transform(X_test)

X_train_processed, X_test_processed

(<Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 168903 stored elements and shape (24129, 82)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 42231 stored elements and shape (6033, 82)>)

In [14]:
from sklearn.ensemble import RandomForestClassifier

# Crear el modelo
modelo = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

# Crear pipeline completo (preprocesamiento + modelo)
pipeline_modelo = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('modelo', modelo)
])

# Entrenar
pipeline_modelo.fit(X_train, y_train)

print("Modelo entrenado correctamente")


Modelo entrenado correctamente


In [15]:
def recomendar_cambios(usuario):
    recomendaciones = []

    # 1. Si trabaja pocas horas
    if usuario['hours.per.week'] < 40:
        recomendaciones.append("Incrementar tus horas semanales podría mejorar tus ingresos.")

    # 2. Educación baja
    niveles_altos = ["Bachelors", "Masters", "Doctorate"]
    if usuario['education'] not in niveles_altos:
        recomendaciones.append("Considera aumentar tu nivel educativo para mejorar tus oportunidades.")

    # 3. Ocupaciones con baja correlación
    ocupaciones_bajas = ["Handlers-cleaners", "Other-service", "Priv-house-serv"]
    if usuario['occupation'] in ocupaciones_bajas:
        recomendaciones.append("Explora ocupaciones técnicas o administrativas con mejores salarios.")

    # 4. País de origen
    if usuario['native.country'] != "United-States":
        recomendaciones.append("La experiencia local puede influir en los ingresos; considera certificaciones locales.")

    # Si no hay recomendaciones
    if len(recomendaciones) == 0:
        recomendaciones.append("Tu perfil ya es competitivo; optimiza tu experiencia laboral.")

    return recomendaciones


In [16]:
usuario_ejemplo = {
    'age': 30,
    'education': 'HS-grad',
    'marital.status': 'Never-married',
    'occupation': 'Other-service',
    'hours.per.week': 35,
    'sex': 'Male',
    'native.country': 'Mexico'
}

recs = recomendar_cambios(usuario_ejemplo)

for r in recs:
    print("-", r)


- Incrementar tus horas semanales podría mejorar tus ingresos.
- Considera aumentar tu nivel educativo para mejorar tus oportunidades.
- Explora ocupaciones técnicas o administrativas con mejores salarios.
- La experiencia local puede influir en los ingresos; considera certificaciones locales.


In [19]:
from sklearn.neighbors import NearestNeighbors

# Entrenar modelo kNN sobre los datos preprocesados
knn = NearestNeighbors(n_neighbors=5, metric='cosine')
knn.fit(X_all)

def recomendar_knn(usuario_dict):
    usuario_df = pd.DataFrame([usuario_dict])
    usuario_vec = pipeline.transform(usuario_df)

    distancias, indices = knn.kneighbors(usuario_vec)

    vecinos = y_all.iloc[indices[0]]

    return vecinos.value_counts(), indices[0]


In [20]:
recs, idx = recomendar_knn(usuario_ejemplo)
print(recs)


income
<=50K    5
Name: count, dtype: int64


In [22]:
usuario_A = {
    'age': 45,
    'education': 'Bachelors',
    'marital.status': 'Married-civ-spouse',
    'occupation': 'Tech-support',
    'hours.per.week': 50,
    'sex': 'Male',
    'native.country': 'United-States'
}


In [26]:
usuario_B = {
    'age': 35,
    'education': 'Some-college',
    'marital.status': 'Married-civ-spouse',
    'occupation': 'Sales',
    'hours.per.week': 40,
    'sex': 'Male',
    'native.country': 'United-States'
}


In [23]:
def recomendar_knn(usuario_dict, k=5):
    usuario_df = pd.DataFrame([usuario_dict])
    usuario_vec = pipeline.transform(usuario_df)

    distancias, indices = knn.kneighbors(usuario_vec, n_neighbors=k)

    vecinos_income = y_all.iloc[indices[0]]

    return vecinos_income.value_counts(), indices[0]


In [24]:
def recomendaciones_trayectoria(usuario_dict, k=5):
    resumen, idx = recomendar_knn(usuario_dict, k=k)

    recomendaciones = []

    # Si la mayoría de vecinos ganan <=50K
    if resumen.get(">50K", 0) < resumen.get("<=50K", 0):
        recomendaciones.append("Tu perfil se parece más a personas que ganan <=50K. Considera mejorar tu trayectoria.")

        # Reglas simples basadas en el dataset
        if usuario_dict['education'] in ['HS-grad', 'Some-college']:
            recomendaciones.append("Aumentar tu nivel educativo a Bachelors o superior podría mejorar tus ingresos.")

        if usuario_dict['hours.per.week'] < 40:
            recomendaciones.append("Trabajar más horas por semana podría acercarte al grupo de mayores ingresos.")

        ocupaciones_bajas = ["Other-service", "Handlers-cleaners", "Priv-house-serv"]
        if usuario_dict['occupation'] in ocupaciones_bajas:
            recomendaciones.append("Cambiar a una ocupación técnica o administrativa podría mejorar tus oportunidades.")

        if usuario_dict['native.country'] != "United-States":
            recomendaciones.append("Certificaciones locales pueden ayudarte a mejorar tu perfil laboral.")

    else:
        recomendaciones.append("Tu perfil es similar al de personas que ganan >50K. Mantén tu trayectoria actual.")

    return recomendaciones


In [25]:
print("Usuario A:")
for r in recomendaciones_trayectoria(usuario_A):
    print("-", r)


Usuario A:
- Tu perfil es similar al de personas que ganan >50K. Mantén tu trayectoria actual.


In [28]:
print("Usuario B:")
for r in recomendaciones_trayectoria(usuario_B):
    print("-", r)


Usuario B:
- Tu perfil se parece más a personas que ganan <=50K. Considera mejorar tu trayectoria.
- Aumentar tu nivel educativo a Bachelors o superior podría mejorar tus ingresos.
